In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Wykorzystanie zasobów obliczeniowych

<span id="usage"></span>
Wykorzystanie zasobów to miara czasu, przez jaki QPU jest zablokowany dla twojego workloadu. Obliczane jest w różny sposób w zależności od używanego trybu wykonania.

* Wykorzystanie Session to czas zegarowy, przez który sesja jest aktywna. Więcej informacji o przejściach między stanami sesji znajdziesz w artykule [Długość sesji](/guides/run-jobs-session#session-length).
* Wykorzystanie Batch to suma czasu kwantowego (czasu spędzonego przez kompleks QPU na przetwarzaniu twojego zadania) wszystkich zadań w batchu.
* Wykorzystanie pojedynczego zadania to czas kwantowy zużyty przez zadanie podczas przetwarzania.

Pamiętaj, że nieudane lub anulowane zadania w pewnych okolicznościach wliczają się do twojego wykorzystania — szczegóły znajdziesz w sekcji [Nieudane i anulowane zadania](#failed-job).

Dla użytkowników płatnych planów wykorzystanie zasobów determinuje koszt workloadu. Szczegóły znajdziesz w artykule [Zarządzanie kosztami](/guides/manage-cost).

<span id="failed-job"></span>
## Wykorzystanie zasobów dla nieudanych i anulowanych zadań
Gdy zadanie zakończy się błędem lub zostanie anulowane, raportowane wykorzystanie wygląda następująco:

* Tryb zadania lub batch: Raportowane wykorzystanie to czas, przez jaki QPU był zablokowany na potrzeby wykonania twojego workloadu, aż do momentu jego niepowodzenia lub anulowania. Jeśli więc błąd lub anulowanie nastąpiło przed zablokowaniem, raportowane wykorzystanie wynosi zero. W przeciwnym razie raportowane wykorzystanie workloadu odpowiada wielkości wykorzystania przed jego niepowodzeniem lub anulowaniem. W rezultacie niektóre nieudane zadania nie pojawiają się w raportowanym wykorzystaniu, a inne tak.

* Tryb Session: Raportowane wykorzystanie to czas zegarowy, przez który sesja jest aktywna, niezależnie od liczby zadań zakończonych błędem lub anulowanych.

<span id="view-usage"></span>
## Sprawdzanie rzeczywistego wykorzystania workloadu
Po zakończeniu workloadu istnieje kilka sposobów na sprawdzenie jego rzeczywistego wykorzystania:

- Uruchom [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) lub [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) w `qiskit-ibm-runtime` w wersji 0.30 lub nowszej. Jeśli używasz starszej wersji `qiskit-ibm-runtime` (>= 0.23 i < 0.30), wykorzystanie nadal można znaleźć w `session.details()["usage_time"]` i `batch.details()["usage_time"]`.
- Użyj [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails), aby zobaczyć wykorzystanie dla konkretnego batchu lub sesji.
- Użyj [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById), aby zobaczyć wykorzystanie dla pojedynczego zadania.

<span id="instance-usage"></span>
## Przeglądanie wykorzystania instancji
Możesz sprawdzić wykorzystanie instancji na stronie [Instancje](https://quantum.cloud.ibm.com/instances) lub — dla osób z odpowiednimi uprawnieniami — na stronie [Analityka](https://quantum.cloud.ibm.com/analytics). Pamiętaj, że strony te mogą wyświetlać różne wartości wykorzystania, ponieważ obliczają je w odmienny sposób.

Strona Instancje pokazuje wykorzystanie w czasie rzeczywistym z ostatnich 28 dni (kroczące), aż do bieżącej chwili w bieżącym dniu. Dane o wykorzystaniu na stronie Analityka są przeliczane co godzinę i obejmują ostatnie 28 pełnych dni, czyli pokazują wykorzystanie od godz. 00:00 sprzed 28 dni do dziś, do pełnej godziny.

## Szacowanie wykorzystania przed przesłaniem zadania
Chociaż dokładne lokalne oszacowanie jest skomplikowane przez dodatkowe operacje wykonywane w ramach tłumienia i łagodzenia błędów, możesz użyć poniższego bazowego wzoru, aby uzyskać przybliżone szacowane wykorzystanie:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` to narzut wynoszący około 2 s na podzadanie. Obejmuje on operacje takie jak ładowanie ładunku do elektroniki sterującej. Twoje zadanie prymitywu może zostać podzielone na wiele podzadań, jeśli jest zbyt duże, aby silnik wykonawczy przetworzył je jednorazowo.
- `rep_delay` to opcja [konfigurowalna przez użytkownika](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay), a jej domyślna wartość jest podawana przez `backend.default_rep_delay`, która na większości backendów IBM Quantum wynosi 250 mikrosekund. Pamiętaj, że zmniejszenie `rep_delay` skraca całkowity czas wykonania QPU, ale kosztem zwiększonego współczynnika błędów przygotowania stanu — więcej informacji znajdziesz w przewodniku [Wykonanie z dynamiczną częstotliwością powtórzeń](/guides/repetition-rate-execution).
- `<circuit length>` to całkowita długość instrukcji. Każda instrukcja zajmuje inną ilość czasu na QPU, więc całkowita długość różni się w zależności od Circuit. Na przykład pomiar może trwać 56 razy dłużej niż bramka `x`. Do znalezienia dokładnego czasu trwania każdej instrukcji można użyć `backend.target[<instruction>][<qubit>].duration`. Typowa długość Circuit wynosi prawdopodobnie od 50 do 100 mikrosekund. Jeśli używasz technik tłumienia lub łagodzenia błędów w prymitywach, do Circuit mogą zostać wstawione dodatkowe instrukcje, co zwiększy całkowitą długość Circuit.
    > **Note:** [Eksperymentalna opcja `scheduler_timing`](/guides/visualize-circuit-timing) zwraca całkowity czas Circuit, ale NIE jest to czas używany do rozliczeń.
- `<num executions>` to całkowita liczba układów pomnożona przez liczbę strzałów, gdzie układy to te wygenerowane po rozgłoszeniu elementów PUB.
    - Jeśli używasz technik łagodzenia błędów w prymitywach, w ramach procesu łagodzenia mogą zostać uruchomione dodatkowe układy, co zwiększy całkowitą liczbę wykonań. Ponadto zaawansowane techniki łagodzenia błędów, takie jak PEA i PEC, wiążą się ze znacznie wyższym narzutem, ponieważ wymagają uruchomienia układów do uczenia szumu.
    - Estimator grupuje obserwable komutujące bitwowo, co zmniejsza liczbę wykonań.

Jeśli nie używasz żadnych zaawansowanych technik łagodzenia błędów ani niestandardowego `rep_delay`, możesz skorzystać z szybkiego wzoru `2+0.00035*<num executions>`.

## Następne kroki
> **Tip:** - Zapoznaj się z tymi wskazówkami: [Minimalizacja czasu wykonywania zadania](minimize-time).
>     - Ustaw [Maksymalny czas wykonania](max-execution-time).
>     - Dowiedz się, jak transpilować lokalnie, w sekcji [Transpiler](./transpile/).
>     - Wypróbuj przewodnik [Porównywanie ustawień Transpilera](/guides/circuit-transpilation-settings).